In [ ]:
import numpy as np
from joblib import load
from loguru import logger
from torch import Tensor
from transformers import BartForConditionalGeneration, BertForSequenceClassification
import plotly.io as pio
pio.renderers.default = "svg" # uncomment this if want interactive renderer
from carl.environment.sokoban.tokenizer import SokobanTokenizer
from carl.environment.sokoban.env import SokobanEnv

from carl.inference_components.conditional_low_level_policy import (
    ConditionalLowLevelPolicy,
    TransformerConditionalLowLevelPolicy,
)
from carl.inference_components.policy import Policy, TransformerPolicy
from carl.inference_components.subgoal_generator import (
    SubgoalGenerator,
    TransformerSubgoalGenerator,
)
from carl.inference_components.value import TransformerValue, Value

logger.info('testing inference components')

# Env and dataset

In [ ]:
import joblib as jl
datapath = './rl-data/validation/sokoban/progress/boards_1000_b4_gs100_c300_p0.35/boards_1000_b4_gs100_c300_p0.35.joblib'
data = jl.load(datapath)

env = SokobanEnv(SokobanTokenizer(None, size_of_board=(12, 12)), num_boxes=4)
env.state_to_repr(data[0])

In [ ]:
logger.info('setting up the environment')
logger.info(
    'It is important to set the categorical distance to 100, because the model was trained with this value. Also the '
    'size of the board should be (12, 12), because the model was trained with this value.'
)
env = SokobanEnv(SokobanTokenizer(cut_distance=None, size_of_board=(12, 12)), num_boxes=4)

# logger.info('setting up the dataset')
path_to_dataset: str = './rl-data/validation/sokoban/offline/12-12-4/sokoban_12_12_4_trajectories_part_88.pkl'
dataset: dict[int, list[np.ndarray]] = load(path_to_dataset)
keys = list(dataset.keys())
current_state: np.ndarray = dataset[keys[1]][1]
display(env.state_to_repr(current_state, title='current state'))
logger.info(
    'state after k is the state after k steps, where k is the number of steps the agent should take. In this '
    'case k = 3.'
)
state_after_k: np.ndarray = dataset[keys[1]][4]
display(env.many_states_to_repr([current_state, state_after_k], titles=['current state', 'state after k=3']))

In [ ]:
from carl.environment.instance_generator import (
    GeneralIterableDataLoader,
    BasicInstanceGenerator,
)

# rl-data/validation/sokoban/offline/12-12-4
path_to_folder_with_data = './rl-data/validation/sokoban/progress/boards_1000_b4_gs25_c300_p0.35'   # path to the folder with trajectories
# its a dict mapping id of trajectory to the list of states in this trajectory
env = SokobanEnv(SokobanTokenizer(None, size_of_board=(12, 12)), num_boxes=4)
instance_generator = BasicInstanceGenerator(
    generator=GeneralIterableDataLoader(path_to_folder_with_data), batch_size=32
)


initial_state_loader = iter(instance_generator.reset_dataloader())
initial_state = next(initial_state_loader).cpu().numpy()
initial_state.shape, type(initial_state), initial_state.dtype

In [ ]:
state = env.set_state(initial_state[0, :])
print(state.shape)
env.state_to_repr(state, 'Example state')

In [ ]:
dataloader = iter(instance_generator.reset_dataloader())
state = next(dataloader).cpu().numpy()[0]
state3 = next(dataloader).cpu().numpy()[0]
env.many_states_to_repr([state, state3], ['test.png', 'test3.png'])

In [ ]:
components_prefix = './rl-data/validation/sokoban/components/full_data/'
from os.path import join

path_to_policy_weights: str = join(components_prefix, 'policy/checkpoint-94820')
path_to_cllp_weights: str = join(components_prefix, 'cllp/8/checkpoint-167585')
path_to_value_function_weights: str = join(components_prefix, 'value/checkpoint-1343100')
path_to_generator_weights: str = join(components_prefix, 'generator/border/8/checkpoint-75856')

# Policy
### CurrentState $\rightarrow$ Action

In [ ]:
current_state.shape

In [ ]:
logger.info('setting up the policy')

policy: Policy = TransformerPolicy(
    policy_network_class=BertForSequenceClassification.from_pretrained,
    path_to_policy_weights=path_to_policy_weights,
    env=env,
    n_actions=2,
)
policy.construct_network()
policy_prediction: Tensor = policy.get_actions(current_state)

logger.info(f'policy prediction - distribution over action: {policy_prediction}')
logger.info(f'Proposed {len(policy_prediction)} actions')
logger.info(f'policy prediction - best action ({policy_prediction[0][0]}) with value {policy_prediction[0][1]}')

In [ ]:
# Lets visualize the policy we have learned

input_state = next(dataloader).cpu().numpy()[0]


visited_states = []
current_state = input_state
for i in range(5):
    visited_states.append(current_state.copy())
    action = policy.get_actions(current_state)
    action = action[0][0]
    logger.info(f'Action: {action}')
    env.set_state(current_state)
    state, _, done, _ = env.step(action)

    current_state = state

    if done:
        break

    if done:
        break

env.many_states_to_repr(
    visited_states, titles=[f'state_{i}' for i in range(len(visited_states) + 1)]
)

# Conditional Low Level Policy
### CurrentState $\times$ SubgoalState $\rightarrow$ Action

In [ ]:
cllp: ConditionalLowLevelPolicy = TransformerConditionalLowLevelPolicy(
    BertForSequenceClassification.from_pretrained, path_to_cllp_weights, env
)
cllp.construct_network()

In [ ]:
current_state: np.ndarray = dataset[keys[1]][1]
state_after_k: np.ndarray = dataset[keys[1]][6]

display(env.many_states_to_repr([current_state, state_after_k], ['current state', 'state after k=5']))

visited_states = []
env.core.restore_full_state_from_np_array_version(current_state)
for i in range(7):
    visited_states.append(current_state.copy())
    if np.array_equal(current_state, state_after_k):
        logger.success('Reached the goal state')
        break
    action = cllp.get_action(current_state, state_after_k)
    action = action.argmax()
    logger.info(f'Action: {action}')
    env.set_state(current_state)
    state, _, done, _ = env.step(action)

    current_state = state


env.many_states_to_repr(
    [*visited_states, state_after_k], [*list(range(len(visited_states))), 'subgoal']
)

# Value function

In [ ]:
import plotly.graph_objects as go

value_function: Value = TransformerValue(
    value_network_class=BertForSequenceClassification.from_pretrained,
    path_to_value_network_weights=path_to_value_function_weights,
    env=env,
    type_of_evaluation='regression',
)

value_function.construct_network()
# value_function_prediction: float = value_function.get_value(current_state)
# solved_state = dataset[keys[1]][-1]
# logger.info(f'value function prediction of unsolved state: {value_function_prediction}')
# logger.info(f'value function prediction of solved state: {value_function.get_value(solved_state)}')
states = [state for state in dataset[keys[11]]]
values = [value_function.get_value(state) for state in states] # can be batched too


fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(len(values))), y=values))
fig.update_layout(title='Value function predictions', xaxis_title='step', yaxis_title='value')
fig.show()

# Goal Generator

In [ ]:
from carl.solver.nodes import SearchTreeNode

logger.info('setting up the subgoal generator')
subgoal_generation_kwargs: dict[str, int] = {
    'num_beams': 8,
    'num_return_sequences': 2,
    'max_new_tokens': 145,
}
generator: SubgoalGenerator = TransformerSubgoalGenerator(
    BartForConditionalGeneration.from_pretrained,
    path_to_generator_weights,
    env,
    subgoal_generation_kwargs,
)
generator.construct_network()

current_state2 = next(dataloader).cpu().numpy()[0]

nodes = [
    SearchTreeNode(
        state=current_state,
        value=0.0,
        low_level_path=[],
        parent_node=None,
        next_expand_with_k_generator=8,
    ),
    SearchTreeNode(
        state=current_state2,
        value=0.0,
        low_level_path=[],
        parent_node=None,
        next_expand_with_k_generator=8,
    ),
]

subgoals_0 = generator.get_subgoals(nodes[0])
subgoals_1 = generator.get_subgoals(nodes[1])

for input_node, subgoals in zip(nodes, [subgoals_0, subgoals_1]):
    fig = env.many_states_to_repr(
        [input_node.state, *[subgoal.state for subgoal in subgoals]],
        ['current_state', *[f'subgoal_proposition_{i}' for i in range(len(subgoals))]],
    )

    display(fig)



In [ ]:
generator.get_network()

# CLLP with Goal Generator - evaluation

In [ ]:
from importlib import reload
import carl.inference_components.cllp_utils as cllp_utils

reload(cllp_utils)

## With generator-generated goals

In [ ]:
display(env.many_states_to_repr([node.state for node in nodes], ['state_0', 'state_1']))
subgoals_first = [subgoal.state for subgoal in subgoals_0]
subgoals_second = [subgoal.state for subgoal in subgoals_1]
env.many_states_to_repr([*subgoals_first, *subgoals_second], ['subgoals_0' for _ in range(len(subgoals_first))] + ['subgoals_1' for _ in range(len(subgoals_second))])

In [ ]:
results = []
for input_node, subgoal_predictions in zip(nodes, [subgoals_0, subgoals_1]):
    cllp_ver_result = cllp_utils.verify_cllp_reaches_subgoals_from_initial_state(
        cllp=cllp,
        goals=[subgoal.state for subgoal in subgoal_predictions],
        initial_state=input_node.state,
        env_creation_fn=lambda: env,
        max_radius=10,
        add_first_batch_to_node_computations=True,
    )
    results.append(cllp_ver_result)
results

In [ ]:
from carl.inference_components.validator import BasicValidator


validator = BasicValidator(env, cllp, 10)

for input_node, result, subgoals in zip(nodes, results, [subgoals_0, subgoals_1]):
    logger.info(f'Verification result for node: {result}')
    for i in range(len(result.paths)):
        trajectory = cllp_utils.trajectory_from_actions(env, input_node.state, result.paths[i])
        subgoal_state = subgoals[i].state
        display(env.many_states_to_repr([input_node.state, subgoal_state, *trajectory],
                                        ['initial_state', 'subgoal', *[f'step_{i}' for i in range(len(trajectory))]]))    

## With small, randomly generated goals

In [ ]:
initial_state = next(dataloader).cpu().numpy()[0]

In [ ]:
def get_goals(
    gen_env: SokobanEnv, gen_initial_state: np.ndarray, num_goals: int = 5, steps_to_goal: int = 7
) -> list[np.ndarray]:
    out_goals = []
    for _ in range(num_goals):
        gen_state = gen_initial_state.copy()
        for _ in range(steps_to_goal):
            action = np.random.randint(0, 4)
            gen_env.set_state(gen_state)
            gen_state, _, done, _ = gen_env.step(action)
            if done:
                break
        out_goals.append(gen_state.copy())
    return out_goals


random_small_goals = get_goals(
    gen_env=env, gen_initial_state=initial_state, num_goals=10, steps_to_goal=3
)

In [ ]:
env.many_states_to_repr(
    [initial_state] + random_small_goals[:5],
    titles=['initial'] + [f'goal_{i}' for i in range(len(random_small_goals[:5]))],
)

In [ ]:
cllp_utils.verify_cllp_reaches_subgoals_from_initial_state(
    cllp=cllp,
    goals=random_small_goals,
    initial_state=initial_state,
    env_creation_fn=lambda: env,
    max_radius=7,
    add_first_batch_to_node_computations=True,
)

----

In [ ]:
# NPuzzle components

from carl.environment.n_puzzle.env import NPuzzleEnv
from carl.environment.n_puzzle.tokenizer import NPuzzleTokenizer

npuzzle_env = NPuzzleEnv(
    tokenizer=NPuzzleTokenizer(size_of_board=(5, 5))
)
npuzzle_env

In [ ]:
npuzzle_env

In [ ]:
import joblib as jl
datapath = './rl-data/validation/npuzzle/progress/fin/fin_puzzles_1000.pkl'
data = jl.load(datapath)

npuzzle_env.state_to_repr(data[0])

In [ ]:
from carl.environment.env import RepresentationType


npuzzle_env.restore_full_state_from_np_array_version(data[0])
npuzzle_env.state_to_repr(npuzzle_env.get_state(), repr_type=RepresentationType.GO_FIGURE)

In [ ]:
from carl.utils.notebook import instantiate_algorithm_from_grid

algo = instantiate_algorithm_from_grid('npuzzle_ada_solve', config_path='../../configs/solve/npuzzle', grid_entry_idx=0)

In [ ]:
from carl.solver.nodes import SearchTreeNode
planner = algo.solver.planner_class(data[0])
planner

In [ ]:
node_k8 = planner.get()
node_k4 = planner.get()
node_others = planner.get()
assert node_others is None

In [ ]:
node_k8.next_expand_with_k_generator, node_k4.next_expand_with_k_generator

In [ ]:
algo.solver.construct_networks()

In [ ]:
algo.solver.subgoal_generator.generator_k_list

In [ ]:
print(algo.solver.subgoal_generator.subgoal_generators[8].sub_generator)

In [ ]:
npuzzle_env.state_to_repr(algo.solver.subgoal_generator.get_subgoals(node_k8)[0].state, repr_type=RepresentationType.GO_FIGURE)

In [ ]:
npuzzle_env.state_to_repr(algo.solver.subgoal_generator.get_subgoals(node_k4)[0].state, repr_type=RepresentationType.GO_FIGURE)

In [ ]:
npuzzle_env.restore_full_state_from_np_array_version(data[0])
npuzzle_env.state_to_repr(npuzzle_env.get_state(), repr_type=RepresentationType.GO_FIGURE)

------

In [1]:
# Rubik'

from carl.utils.notebook import instantiate_algorithm_from_grid

algo = instantiate_algorithm_from_grid('rubik_ada_solve', config_path='../../configs/solve/rubik', grid_entry_idx=0)


/home/mtyrolski/dev/carl/CaRL/carl/utils/notebook.py:22: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  initialize(config_path=config_path)
2025-07-08 23:28:39.155 | INFO     | carl.slurm.grid_search:validate_config:48 - Validating carl_grid syntax.
2025-07-08 23:28:39.168 | INFO     | carl.slurm.grid_search:validate_config:51 - Validating cartesian entry: {'subgoal_generator.generator_k_list': [[4, 2]], 'subgoal_generator.paths_to_generator_weights': [['./rl-data/validation/rubik/components/shuffle_70/generator/4/checkpoint-270710', './rl-data/validation/rubik/components/shuffle_70/generator/2/checkpoint-270710']], 'validator.cllp.path_to_conditional_low_level_policy_weights': ['./rl-data/validation/rubik/components/shuffle_70/cllp/4/checkpoint-141063'], 'algorithm.result_logger.custom_logger.tags': [['inner_eval', 'rubik', 'AdaSubS', 'solve', 'k_42', 'shuffle_general']], 'subgoal

Using Neptune API token: eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiJmYmIyNDk2Yi1kMmRiLTRhYmItYTRlMy00NjIyNzViN2Y3ZDAifQ==
[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/mtyrol/drop/e/DROP-705


2025-07-08 23:28:51.081 | INFO     | carl.algorithms.solve_instances:__init__:43 - Using 1 parallel workers


In [2]:
import joblib

ds = joblib.load('./rl-data/validation/rubik/progress/shuffle_general/rubik_eval_data_part_shuffle_1000_0.pkl')
ds[:3]

['gbbgywbwoyorybgrogyowrrbrrobryoggwgworoyobrbbyygywwwwg',
 'orwwyrryowogwbbrrgybborbygbyybwgyrwrobgoogbgwoywgwrgoy',
 'ogbryrwyrwbgbbgbwwroywrwbrgbgorgyybywoggoygoyryoowwobr']

In [3]:
algo.solver.construct_networks()

2025-07-08 23:28:51.135 | DEBUG    | carl.inference_components.component:instantiate_network:60 - Loading weights from ./rl-data/validation/rubik/components/shuffle_70/generator/4/checkpoint-270710
2025-07-08 23:28:51.138 | INFO     | carl.inference_components.component:instantiate_network:82 - Provided direct path to the ckpt
2025-07-08 23:28:51.389 | SUCCESS  | carl.inference_components.component:instantiate_network:94 - Loaded weights from ./rl-data/validation/rubik/components/shuffle_70/generator/4/checkpoint-270710
2025-07-08 23:28:51.393 | DEBUG    | carl.inference_components.component:instantiate_network:60 - Loading weights from ./rl-data/validation/rubik/components/shuffle_70/generator/2/checkpoint-270710
2025-07-08 23:28:51.394 | INFO     | carl.inference_components.component:instantiate_network:82 - Provided direct path to the ckpt
2025-07-08 23:28:51.633 | SUCCESS  | carl.inference_components.component:instantiate_network:94 - Loaded weights from ./rl-data/validation/rubik/

In [4]:
root_state = 'gbbgywbwoyorybgrogyowrrbrrobryoggwgworoyobrbbyygywwwwg'

In [5]:
algo.solver.planner_class

functools.partial(<class 'carl.planners.adasubs.AdasubsPlanner'>, generators_k_list=[4, 2])

In [6]:

from carl.planners.adasubs import AdasubsPlanner


planner: AdasubsPlanner = algo.solver.planner_class(root_state)
len(planner.nodes_queue)

2

In [7]:
retrieved_nodes = []
for _ in range(3):
    node = planner.get()
    retrieved_nodes.append(node)
    if node is None:
        break

retrieved_nodes

[SearchTreeNode(state.shape=(54,), value=0, next_expand_k=4),
 SearchTreeNode(state.shape=(54,), value=0, next_expand_k=2),
 None]

In [8]:
subgoals_k8 = algo.solver.subgoal_generator.get_subgoals(retrieved_nodes[0])
subgoals_k4 = algo.solver.subgoal_generator.get_subgoals(retrieved_nodes[1])

In [9]:
from carl.inference_components.cllp_utils import verify_cllp_reaches_subgoals_from_initial_state
verify_cllp_reaches_subgoals_from_initial_state(
    cllp=algo.solver.validator.cllp,
    goals=[subgoal.state for subgoal in subgoals_k4],
    initial_state=retrieved_nodes[1].state,
    env_creation_fn=lambda: algo.solver.subgoal_generator.env,
    max_radius=10,
)

TypeError: string indices must be integers, not 'tuple'

---- 

In [ ]:
from carl.environment.gym_rubik.rubik_env import RubikEnv
from carl.environment.gym_rubik.tokenizer import RubikCubeTokenizer
rubik_env = RubikEnv(RubikCubeTokenizer())

state_before_set = rubik_env.get_state()
state_before_set

In [ ]:
rubik_env.restore_full_state_from_np_array_version(root_state)
state_after_set = rubik_env.get_state()
state_after_set

In [ ]:
# s == cube_state_to_str(cube_str_to_state(s))


for state in [state_before_set, state_after_set]:
    assert state == rubik_env.cube_state_to_str(rubik_env.cube_str_to_state(state))



In [ ]:
# Debug: Compare current state and subgoal state strings for Rubik's Cube

# Example: Use the first subgoal from subgoals_k8 (adjust as needed for your setup)
if len(subgoals_k8) > 0:
    subgoal_state = subgoals_k8[0].state
    current_state = retrieved_nodes[0].state
    print('Current state:', current_state)
    print('Subgoal state:', subgoal_state)
    print('Are they string-equal?', current_state == subgoal_state)
    # Optionally, visualize both states if you have a visualization function
    # print(rubik_env.state_to_repr(current_state, title='Current'))
    # print(rubik_env.state_to_repr(subgoal_state, title='Subgoal'))
else:
    print('No subgoals found to compare.')


In [ ]:
# Try all possible actions from the current state and print resulting states
from carl.environment.gym_rubik.rubik_env import ACTION_LOOKUP

if len(subgoals_k8) > 0:
    subgoal_state = subgoals_k8[0].state
    current_state = retrieved_nodes[0].state
    print('Current state:', current_state)
    print('Subgoal state:', subgoal_state)
    print('Trying all actions from current state:')
    for action_idx in range(len(ACTION_LOOKUP)):
        # Reset environment to current state
        rubik_env.restore_full_state_from_np_array_version(current_state)
        # Take action
        next_state, _, _, _ = rubik_env.step(action_idx)
        print(f'Action {action_idx} ({rubik_env.action_name(action_idx)}): {next_state}')
        if next_state == subgoal_state:
            print(f'--> Subgoal reached with action {action_idx}!')
else:
    print('No subgoals found to test actions.')


### Verification that the CLLP work without generator - so by artificially extracted goals from trajectories

In [ ]:
from carl.inference_components.cllp_utils import CLLPVerificationResult

import numpy as np
trajectories = joblib.load('./rl-data/validation/rubik/offiline/shuffle_100/rubik_offline_data_part_1.pkl')
first_key = next(iter(trajectories.keys()))
selected_trajectory = trajectories[first_key]
L = len(selected_trajectory)
N = 100


random_ids_begins = np.random.randint(0, L - N, size=N)
random_shifts = np.random.randint(1, 5, size=N)

hit_rates: list[CLLPVerificationResult] =| []
for j in range(N):
    begin = random_ids_begins[j]
    shift = random_shifts[j]
    is_reachable = verify_cllp_reaches_subgoals_from_initial_state(
        cllp=algo.solver.validator.cllp,
        goals=[selected_trajectory[begin + shift]],
        initial_state=selected_trajectory[begin],
        env_creation_fn=lambda: algo.solver.subgoal_generator.env,
        max_radius=5,
    )
    hit_rates.append(is_reachable)
    
hit_rate = sum([hr.success_rate for hr in hit_rates]) / len(hit_rates)
hit_rate

### Verification that artificially extracted goals from trajectories are reachable

In [ ]:
from itertools import product
from loguru import logger

reachable_one_step_away = 0
unreachable_one_step_away = 0
reachable_two_steps_away = 0
unreachable_two_steps_away = 0
for idx_begin in random_ids_begins:
    starting_state = selected_trajectory[idx_begin] # k0
    first_subgoal = selected_trajectory[idx_begin + 1] # k1
    second_subgoal = selected_trajectory[idx_begin + 2] # k2
    
    # try hard all possible actions without CLLP
    for action_idx in range(len(ACTION_LOOKUP)):
        # Reset environment to first subgoal
        rubik_env.restore_full_state_from_np_array_version(starting_state)
        # Take action
        next_state, _, _, _ = rubik_env.step(action_idx)
        
        if next_state == second_subgoal:
            reachable_one_step_away += 1
            break
    if next_state != second_subgoal:
        unreachable_one_step_away += 1
    
    # try hard second subgoal
    
    for (a1, a2) in product(range(len(ACTION_LOOKUP)), repeat=2):
        # Reset environment to first subgoal
        rubik_env.restore_full_state_from_np_array_version(starting_state)
        # Take action
        state_after_one_step, _, _, _ = rubik_env.step(a1)
        # Take second action
        state_after_two_steps, _, _, _ = rubik_env.step(a2)
        if state_after_two_steps == second_subgoal:
            reachable_two_steps_away += 1
            break

    if state_after_two_steps != second_subgoal:
        unreachable_two_steps_away += 1
logger.info(f'One step away reachable: {reachable_one_step_away}, unreachable: {unreachable_one_step_away}')
logger.info(f'Two steps away reachable: {reachable_two_steps_away}, unreachable: {unreachable_two_steps_away}')

# rates
logger.info(
    f'One step away reachable rate: {reachable_one_step_away / (reachable_one_step_away + unreachable_one_step_away)}')
logger.info(
    f'Two steps away reachable rate: {reachable_two_steps_away / (reachable_two_steps_away + unreachable_two_steps_away)}'
)
    

In [ ]:
# Deep diagnostic: For a sample of transitions, print all one- and two-step results and compare to dataset next state
from collections import Counter

def hamming_distance(s1, s2):
    return sum(c1 != c2 for c1, c2 in zip(s1, s2))

sample_size = 5
print('Deep diagnostic of state transitions:')
for idx in range(sample_size):
    begin = random_ids_begins[idx]
    start_state = selected_trajectory[begin]
    next_state = selected_trajectory[begin + 1]
    print(f'\nSample {idx+1}:')
    print('Start state: ', start_state)
    print('Next state (dataset):', next_state)
    found = False
    # Try all one-step actions
    one_step_results = []
    for action_idx in range(len(ACTION_LOOKUP)):
        rubik_env.restore_full_state_from_np_array_version(start_state)
        result, _, _, _ = rubik_env.step(action_idx)
        one_step_results.append(result)
        if result == next_state:
            print(f'  One-step match: action {action_idx} ({rubik_env.action_name(action_idx)})')
            found = True
    if not found:
        print('  No one-step action produces the next state.')
        # Show closest one-step result by Hamming distance
        distances = [hamming_distance(result, next_state) for result in one_step_results]
        min_dist = min(distances)
        print(f'  Closest one-step result is {min_dist} moves away:')
        for i, d in enumerate(distances):
            if d == min_dist:
                print(f'    Action {i} ({rubik_env.action_name(i)}): {one_step_results[i]}')
    # Try all two-step actions
    found2 = False
    two_step_results = []
    for a1, a2 in product(range(len(ACTION_LOOKUP)), repeat=2):
        rubik_env.restore_full_state_from_np_array_version(start_state)
        s1, _, _, _ = rubik_env.step(a1)
        s2, _, _, _ = rubik_env.step(a2)
        two_step_results.append((a1, a2, s2))
        if s2 == next_state:
            print(f'  Two-step match: actions {a1},{a2} ({rubik_env.action_name(a1)}, {rubik_env.action_name(a2)})')
            found2 = True
            break
    if not found2:
        print('  No two-step action sequence produces the next state.')
        # Show closest two-step result by Hamming distance
        distances2 = [hamming_distance(s2, next_state) for _, _, s2 in two_step_results]
        min_dist2 = min(distances2)
        print(f'  Closest two-step result is {min_dist2} moves away:')
        for (a1, a2, s2), d in zip(two_step_results, distances2):
            if d == min_dist2:
                print(f'    Actions {a1},{a2} ({rubik_env.action_name(a1)}, {rubik_env.action_name(a2)}): {s2}')


In [ ]:
# Deep debug: Check round-trip serialization and action effects for Rubik's Cube states

from carl.environment.gym_rubik.rubik_env import RubikEnv, ACTION_LOOKUP
from carl.environment.gym_rubik.tokenizer import RubikCubeTokenizer

rubik_env = RubikEnv(RubikCubeTokenizer())

# 1. Check round-trip serialization for a batch of states from the dataset
print("Checking round-trip serialization for first 10 states in the trajectory:")
for i, s in enumerate(selected_trajectory[:10]):
    s2 = rubik_env.cube_state_to_str(rubik_env.cube_str_to_state(s))
    print(f"State {i}:")
    print("  Original:   ", s)
    print("  Round-trip: ", s2)
    print("  Match:", s == s2)
    if s != s2:
        print("  MISMATCH FOUND!")
    print()

# 2. For a sample state, apply all actions and print resulting state strings
sample_idx = 0
sample_state = selected_trajectory[sample_idx]
print(f"Applying all actions to state {sample_idx}: {sample_state}")
for action_idx in range(len(ACTION_LOOKUP)):
    rubik_env.restore_full_state_from_np_array_version(sample_state)
    next_state, _, _, _ = rubik_env.step(action_idx)
    print(f"  Action {action_idx} ({rubik_env.action_name(action_idx)}): {next_state}")

# 3. Compare to the next state in the trajectory
if len(selected_trajectory) > sample_idx + 1:
    print("\nNext state in trajectory:", selected_trajectory[sample_idx + 1])
    print("Is any action result equal to next state?")
    found = False
    for action_idx in range(len(ACTION_LOOKUP)):
        rubik_env.restore_full_state_from_np_array_version(sample_state)
        next_state, _, _, _ = rubik_env.step(action_idx)
        if next_state == selected_trajectory[sample_idx + 1]:
            print(f"  YES: Action {action_idx} ({rubik_env.action_name(action_idx)}) produces the next state.")
            found = True
    if not found:
        print("  No single action produces the next state. Try two-step sequences for further debugging if needed.")

In [ ]:
# Diagnostic: Print action effects on solved state and compare to dataset transitions

from carl.environment.gym_rubik.rubik_env import RubikEnv, ACTION_LOOKUP
from carl.environment.gym_rubik.tokenizer import RubikCubeTokenizer

rubik_env = RubikEnv(RubikCubeTokenizer())

# Get a solved state string according to the environment
solved_state = rubik_env.cube_state_to_str(rubik_env.cube_str_to_state(rubik_env.cube_state_to_str(rubik_env.cube_str_to_state(''.join(['w']*9 + ['r']*9 + ['b']*9 + ['o']*9 + ['g']*9 + ['y']*9)))))
print('Solved state (env convention):', solved_state)

# Print all action names and their effect on the solved state
print('\nAction mapping and effect on solved state:')
for action_idx, action_name in enumerate(ACTION_LOOKUP):
    rubik_env.restore_full_state_from_np_array_version(solved_state)
    next_state, _, _, _ = rubik_env.step(action_idx)
    print(f'  Action {action_idx} ({rubik_env.action_name(action_idx)}): {next_state}')

# If you have a solved state from the dataset, compare it here
if len(selected_trajectory) > 0:
    dataset_solved_state = selected_trajectory[0]
    print('\nFirst state in dataset (possibly solved):', dataset_solved_state)
    print('Is env solved state == dataset solved state?', solved_state == dataset_solved_state)
    print('Hamming distance:', sum(a != b for a, b in zip(solved_state, dataset_solved_state)))
    print('\nEffect of all actions on dataset solved state:')
    for action_idx, action_name in enumerate(ACTION_LOOKUP):
        rubik_env.restore_full_state_from_np_array_version(dataset_solved_state)
        next_state, _, _, _ = rubik_env.step(action_idx)
        print(f'  Action {action_idx} ({rubik_env.action_name(action_idx)}): {next_state}')


In [ ]:
# Analyze dataset state string convention and propose conversion
from collections import Counter
import numpy as np

# 1. Analyze color distribution and facelet positions in the dataset's first state
if len(selected_trajectory) > 0:
    dataset_solved_state = selected_trajectory[0]
    print('Dataset first state:', dataset_solved_state)
    print('Length:', len(dataset_solved_state))
    print('Color counts:', Counter(dataset_solved_state))
    print('Facelet positions (first 9, next 9, ...):')
    for i in range(0, 54, 9):
        print(f'  Face {i//9}:', dataset_solved_state[i:i+9])

    # Try to infer face color by majority in each block
    print('\nLikely face color for each face:')
    for i in range(0, 54, 9):
        block = dataset_solved_state[i:i+9]
        most_common = Counter(block).most_common(1)[0][0]
        print(f'  Face {i//9}: {most_common}')

# 2. Propose a conversion function (template)
def convert_env_to_dataset_convention(env_state_str):
    """
    Convert an environment state string to the dataset's convention.
    You must fill in the correct mapping based on the analysis above.
    """
    # Example: permute faces and/or facelets, or remap colors
    # This is a placeholder. You must determine the correct permutation!
    # For example, if dataset order is [B, U, Y, G, W, O] and env is [U, R, F, D, L, B]:
    # mapping = [facelet indices in env order for each dataset facelet]
    mapping = np.arange(54)  # TODO: fill in with correct permutation
    env_chars = list(env_state_str)
    return ''.join([env_chars[i] for i in mapping])

# 3. Next steps (printed as instructions)
print("\nNext steps:")
print("1. Manually inspect the facelet blocks above and compare to your environment's solved state blocks.")
print("2. Determine the face order and color mapping used in the dataset.")
print("3. Fill in the 'mapping' array in the conversion function to permute env state strings to dataset convention.")
print("4. Use this function to convert all env states before comparing or using with the dataset.")
print("5. If color letters differ, add a color remapping step as well.")